# 5X72 original rigid geometry: saved construction and scores

The [prespecified workflow](5x72_rigid_workflow.md) places one conformer per ligand at two experimental heavy torsions, then fits its24 heavy atoms while moving all39 atoms. Original rigid geometry **including all H** stays unchanged. This notebook reads saved evidence: **no new molecular conformer, preparation, scoring, docking or optimization**. It independently recomputes numerical distances and the fit to verify the captured coordinates, without producing another molecular input. #17/#6/#33 stay partial/open. The earlier reference used generatedH; a pure heavy-coordinate causal effect is not isolated.

In [1]:
import gzip, json, sys
from pathlib import Path
checkout=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'devtools/qualify_5x72_rigid.py').is_file())
sys.path.insert(0,str(checkout))
import numpy as np
import pandas as pd
import pyunitwizard as puw
import dockingmt as dmt
from devtools.qualify_5x72_rigid import OUTPUT,sha,load_archives,load_reference,compare_saved,score_components
payload=OUTPUT.read_bytes();record=json.loads(gzip.decompress(payload))
checkpoint=json.loads(OUTPUT.with_name('checkpoint_2026-10-09.json').read_text())
assert sha(payload)==checkpoint['producer_archive']['gzip_sha256']
assert sha(OUTPUT.with_name('producer_2026-10-09.py.txt').read_bytes())==record['consumer_source_sha256']['devtools/qualify_5x72_rigid.py']
assert record['new_searches']==0 and record['new_fixed_evaluations']==8
print('Original gzip SHA256:',sha(payload))
print('Preserved provider source:',record['qualified_provider_commit'])
summary=[]
for ligand,conformer in record['conformers'].items():
    for context,data in conformer['fixed_scores']['native'].items():
        summary.append({'ligand':ligand,'score_receptor':context,
            'heavy_positional_rmsd_angstrom':conformer['construction']['geometry']['heavy_positional_rmsd_angstrom'],
            'prepared_total_kcal_mol':score_components(data)['fixed'],
            'prepared_minus_experimental_kcal_mol':conformer['prepared_minus_experimental_reference_kcal_mol']['native'][context]['fixed']})
    assert conformer['root_order_agrees_at_0_001_kcal_mol']
    assert all(v==0 for values in conformer['root_order_component_difference_kcal_mol'].values() for v in values.values())
display(pd.DataFrame(summary))

Original gzip SHA256: f0e91b99793bc61403aedf660016f532e1a46c5815f92441f511dbac8ffb4179
Preserved provider source: 5bd893c85fe8d211663b2b1f865f5f1d2c382a90


,ligand,score_receptor,heavy_positional_rmsd_angstrom,prepared_total_kcal_mol,prepared_minus_experimental_kcal_mol
0,p59,occupied,0.513138,-8.551,1.983
1,p59,sham,0.513138,-7.618,2.043
2,p69,occupied,0.679452,-2.667,7.832
3,p69,sham,0.679452,-1.666,7.960


The following independently checks every full-atom pair,246 within-fragment distances and42 bonds per ligand, the captured proper fit and coordinate handedness. The near-reference RMSD describes a constructed point, not a returned pose or global optimum over torsions. Small RMSD does not guarantee favorable local interactions.

In [2]:
checked_pairs=0; geometry_summary=[]
for ligand,conformer in record['conformers'].items():
    construction=conformer['construction'];geometry=construction['geometry']
    source=np.asarray(geometry['full_atom_coordinates_angstrom']['original'])
    fitted=np.asarray(geometry['full_atom_coordinates_angstrom']['fitted'])
    reference=np.asarray(geometry['full_atom_coordinates_angstrom']['experimental_heavy_generated_H_reference'])
    assert len(geometry['pairs'])==741
    assert sum(p['within_fragment'] for p in geometry['pairs'])==246
    assert sum(p['bond'] for p in geometry['pairs'])==42
    for pair in geometry['pairs']:
        i,j=pair['source_atom_pair']
        a=np.linalg.norm(source[i]-source[j]);b=np.linalg.norm(fitted[i]-fitted[j]);c=np.linalg.norm(reference[i]-reference[j])
        np.testing.assert_allclose([a,b,c,b-a],[pair['original_distance_angstrom'],pair['fitted_distance_angstrom'],pair['experimental_reference_distance_angstrom'],pair['fitted_minus_original_angstrom']],rtol=0,atol=1e-9)
        if pair['within_fragment'] or pair['bond']:assert abs(b-a)<=1e-9
        checked_pairs+=1
    snapshot=construction['adjusted_snapshot']
    assert snapshot['structure_units']['coordinates']=='nm'
    adjusted=puw.get_value(puw.quantity(snapshot['structures']['coordinates'],snapshot['structure_units']['coordinates']),to_unit='angstrom')[0]
    a,b=adjusted[:24],reference[:24];ca,cb=a.mean(axis=0),b.mean(axis=0)
    u,s,vt=np.linalg.svd((a-ca).T@(b-cb));rotation=u@np.diag([1,1,np.linalg.det(u@vt)])@vt
    assert abs(np.linalg.det(rotation)-1)<1e-10
    np.testing.assert_allclose((adjusted-ca)@rotation+cb,fitted,rtol=0,atol=1e-9)
    volumes=[np.linalg.det(x[[6,8,17]]-x[28]) for x in [source,fitted]]
    assert np.sign(volumes[0])==np.sign(volumes[1])
    np.testing.assert_allclose(volumes,geometry['original_and_fitted_handed_volume_angstrom_cubed'],rtol=0,atol=1e-9)
    rmsd=np.sqrt(np.mean(np.sum((fitted[:24]-reference[:24])**2,axis=1)))
    assert abs(rmsd-geometry['heavy_positional_rmsd_angstrom'])<1e-9
    geometry_summary.append({'ligand':ligand,'pairs':741,'within_fragment':246,'bonds':42,'heavy_positional_rmsd_angstrom':rmsd})
assert checked_pairs==1482
display(pd.DataFrame(geometry_summary))

,ligand,pairs,within_fragment,bonds,heavy_positional_rmsd_angstrom
0,p59,741,246,42,0.513138
1,p69,741,246,42,0.679452


All eight captured constructed-geometry scores, their component contrasts to the earlier experimental-reference scores and all312 saved pose/context comparisons follow. Intra equals best-pose intra in these unchanged fixed evaluations, so its sign alone does not explain total ordering. The large P69 contrast is mainly in ligand-inter, without identifying a physical contact or selecting a remedy.

In [3]:
archives=load_archives()[0];references=load_reference();rows=[];cells=[];evaluations=0
with puw.context(standard_units=['pm','fs','K','mole','dalton','e','kJ/mol','degrees']):
    for ligand,conformer in record['conformers'].items():
        for order,contexts in conformer['fixed_scores'].items():
            original=dmt.DockingPose.from_dict(conformer['poses'][order])
            assert original.rank is None and original.scores=={}
            for context,data in contexts.items():
                scored=dmt.DockingPose.from_dict(data);history=scored.metadata['scoring_history'][-1]
                assert dmt.verify_captured_inputs(history['backend_artifacts'])
                assert history['operation']=='score' and len(history['scores'])==8
                np.testing.assert_array_equal(puw.get_value(scored.coordinates,to_unit='angstrom'),puw.get_value(original.coordinates,to_unit='angstrom'))
                contrast={key:value-score_components(references['references'][ligand]['fixed_scores'][order][context])[key] for key,value in history['scores'].items()}
                assert contrast==conformer['prepared_minus_experimental_reference_kcal_mol'][order][context]
                evaluations+=1
        compared,summarized=compare_saved(ligand,archives[ligand],conformer['fixed_scores'])
        rows.extend(compared);cells.extend(summarized)
assert evaluations==8 and len(rows)==312 and len(cells)==96
assert rows==record['comparison_rows'] and cells==record['cell_summaries']
table=pd.DataFrame(rows);table['pose_below_constructed']=[r['pose_minus_reference_kcal_mol']['fixed']<0 for r in rows]
display(table.groupby(['ligand','search_receptor','score_receptor']).agg(saved_poses=('original_rank','size'),below_constructed=('pose_below_constructed','sum')))
print('8 original captured scores and all156 retained poses /312 saved contexts verified; no new historical evaluation.')

saved_poses  below_constructed
ligand search_receptor score_receptor                                
p59    occupied        occupied                 28                 25
                       sham                     28                 25
       sham            occupied                 50                  0
                       sham                     50                 50
p69    occupied        occupied                 29                 29
                       sham                     29                 29
       sham            occupied                 49                  9
                       sham                     49                 49

8 original captured scores and all156 retained poses /312 saved contexts verified; no new historical evaluation.
